In [0]:
%run "/Databricks Masterclass/Exploring Databricks"

### Hardcoded values

+--------+---+--------------+
|    name|age|           job|
+--------+---+--------------+
|Varshini| 24|  Data Analyst|
|   Varun| 25|Data Scientist|
|    Rana| 27| Data Engineer|
+--------+---+--------------+



### Access ADLS Using Databricks

#### Create an app, get the app_id and tenant_id. Then create a secret and copy it. finally, go to the storage acccount and assing a blob contributor data to the app.

### Dbutils

### 1. dbutils.fs

[FileInfo(path='abfss://source@adlsfordbs.dfs.core.windows.net/Sales (1).csv', name='Sales (1).csv', size=869537, modificationTime=1743431275000)]

### 2. dbutils.widgets - text & get

# Delta Lake

In [0]:
df.write.format('delta')\
         .mode('append')\
         .option('path', 'abfss://destination@adlsfordbs.dfs.core.windows.net/Sales')\
         .save()

### Managed Vs External Delta Table

### **Database**

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS SalesDB;

### **Managed table**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS SalesDB.managedtbl
(
  id INT,
  name STRING,
  marks INT
)
USING DELTA 

In [0]:
%sql
INSERT INTO salesdb.managedtbl
values 
(1, 'aa', 35),
(2, 'bb', 33),
(3, 'cc', 37),
(4, 'dd', 40)

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
select * from salesdb.managedtbl

id,name,marks
1,aa,35
2,bb,33
3,cc,37
4,dd,40
1,aa,35
2,bb,33
3,cc,37
4,dd,40
1,aa,35
2,bb,33


### External Table

In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsfordbs.dfs.core.windows.net",
    dbutils.secrets.get(scope="subashscope", key="app-secret"))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS SalesDB.externaltbl
(
  id INT,
  name STRING,
  marks INT
)
USING DELTA
LOCATION 'abfss://destination@adlsfordbs.dfs.core.windows.net/tables/salesdb/externaltbl/';

In [0]:
%sql
INSERT INTO SalesDB.externaltbl
VALUES (1, 'aa', 24),
(2, 'bb', 32), (3, 'cc', 43)

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from SalesDB.externaltbl

id,name,marks
1,aa,24
2,bb,32
3,cc,43
4,dd,54
5,ee,43
6,ff,41
7,gg,23
1,aa,24
2,bb,32
3,cc,43


### Delta Table Functionalities

### **Insert**

In [0]:
%sql
insert into SalesDB.externaltbl
values(4, 'dd', 54),
(5, 'ee', 43),
(6, 'ff', 41),
(7, 'gg', 23)

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
select * from SalesDB.externaltbl

id,name,marks
1,aa,24
2,bb,32
3,cc,43
4,dd,54
5,ee,43
6,ff,41
7,gg,23
4,dd,54
5,ee,43
6,ff,41


In [0]:
%sql
DELETE FROM SalesDB.externaltbl
where id = 7

num_affected_rows
2


### Data Versioning

In [0]:
%sql
DESCRIBE HISTORY SalesDB.externaltbl

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
17,2025-04-01T15:04:17Z,2676310242579246,ownpurposexd@gmail.com,DELETE,"Map(predicate -> [""(id#4769 = 7)""])",null,List(3575137121811217),0331-144109-ocxvto0l,16,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5357, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 2124, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 3177)",null,Databricks-Runtime/15.4.x-scala2.12
16,2025-04-01T15:04:05Z,2676310242579246,ownpurposexd@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3575137121811217),0331-144109-ocxvto0l,15,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1105)",null,Databricks-Runtime/15.4.x-scala2.12
15,2025-04-01T15:03:58Z,2676310242579246,ownpurposexd@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3575137121811217),0331-144109-ocxvto0l,14,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1091)",null,Databricks-Runtime/15.4.x-scala2.12
14,2025-04-01T12:53:25Z,2676310242579246,ownpurposexd@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3575137121811217),0331-144109-ocxvto0l,13,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2196, p25FileSize -> 1145, numDeletionVectorsRemoved -> 0, minFileSize -> 1145, numAddedFiles -> 1, maxFileSize -> 1145, p75FileSize -> 1145, p50FileSize -> 1145, numAddedBytes -> 1145)",null,Databricks-Runtime/15.4.x-scala2.12
13,2025-04-01T12:46:19Z,2676310242579246,ownpurposexd@gmail.com,RESTORE,"Map(version -> 2, timestamp -> null)",null,List(3575137121811217),0331-144109-ocxvto0l,12,Serializable,false,"Map(numRestoredFiles -> 2, removedFilesSize -> 1261, numRemovedFiles -> 1, restoredFilesSize -> 2196, numOfFilesAfterRestore -> 2, tableSizeAfterRestore -> 2196)",null,Databricks-Runtime/15.4.x-scala2.12
12,2025-04-01T12:46:01Z,2676310242579246,ownpurposexd@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3575137121811217),0331-144109-ocxvto0l,11,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3446, p25FileSize -> 1261, numDeletionVectorsRemoved -> 1, minFileSize -> 1261, numAddedFiles -> 1, maxFileSize -> 1261, p75FileSize -> 1261, p50FileSize -> 1261, numAddedBytes -> 1261)",null,Databricks-Runtime/15.4.x-scala2.12
11,2025-04-01T12:45:53Z,2676310242579246,ownpurposexd@gmail.com,DELETE,"Map(predicate -> [""(id#4225 = 7)""])",null,List(3575137121811217),0331-144109-ocxvto0l,10,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3981, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1709, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 2248)",null,Databricks-Runtime/15.4.x-scala2.12
10,2025-04-01T12:45:45Z,2676310242579246,ownpurposexd@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3575137121811217),0331-144109-ocxvto0l,9,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1105)",null,Databricks-Runtime/15.4.x-scala2.12
9,2025-04-01T12:45:40Z,2676310242579246,ownpurposexd@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3575137121811217),0331-144109-ocxvto0l,8,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1091)",null,Databricks-Runtime/15.4.x-scala2.12
8,2025-04-01T12:10:48Z,2676310242579246,ownpurposexd@gmail.com,OPTIMIZE,"Map(predi

### Time Travel

In [0]:
%sql
select * from SalesDB.externaltbl

id,name,marks
1,aa,24
2,bb,32
3,cc,43
4,dd,54
5,ee,43
6,ff,41
4,dd,54
5,ee,43
6,ff,41
1,aa,24


In [0]:
%sql
select * from SalesDB.externaltbl version as of 2

id,name,marks
4,dd,54
5,ee,43
6,ff,41
7,gg,23
1,aa,24
2,bb,32
3,cc,43


In [0]:
%sql
RESTORE TABLE SalesDB.externaltbl to version as of 2;

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

### Vacuum -- Default retention rate is 7 days

In [0]:
%sql
--Vacuum SalesDB.externaltbl

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

In [0]:
%sql
--vacuum SalesDB.externaltbl retain 0 hours

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

### Delta table optimization

### **optimize**

In [0]:
%sql
optimize SalesDB.externaltbl

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

In [0]:
%sql
select * from SalesDB.externaltbl

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

### **ZORDER BY**

In [0]:
%sql
optimize SalesDB.externaltbl ZORDER BY (id)

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

In [0]:
%sql
select * from salesdb.externaltbl

io.delta.exceptions.ConcurrentWriteException: [DELTA_CONCURRENT_WRITE] ConcurrentWriteException: A concurrent transaction has written new data since the current transaction read the table. Please try the operation again.
Conflicting commit: {"timestamp":1743519868111,"userId":"2676310242579246","userName":"ownpurposexd@gmail.com","operation":"OPTIMIZE","operationParameters":{"predicate":[],"auto":true,"clusterBy":[],"zOrderBy":[],"batchId":0},"notebook":{"notebookId":"3575137121811217"},"clusterId":"0331-144109-ocxvto0l","readVersion":17,"isolationLevel":"SnapshotIsolation","isBlindAppend":false,"operationMetrics":{"numRemovedFiles":"3","numRemovedBytes":"3341","p25FileSize":"1250","numDeletionVectorsRemoved":"2","minFileSize":"1250","numAddedFiles":"1","maxFileSize":"1250","p75FileSize":"1250","p50FileSize":"1250","numAddedBytes":"1250"},"tags":{"delta.rowTracking.preserved":"false"},"engineInfo":"Databricks-Runtime/15.4.x-scala2.12","txnId":"abff143f-1b93-46ba-b793-f2f140f52488"}
Ref

### **Incremental Data Loading Using AutoLoader**

In [0]:
df = spark.readStream.format('cloudFiles')\
                     .option('cloudFiles.format', 'parquet')\
                     .option('cloudFiles.schemaLocation', 'abfss://aldestination@adlsfordbs.dfs.core.windows.net/schemalocation')\
                     .load('abfss://alsource@adlsfordbs.dfs.core.windows.net')

In [0]:
df.writeStream.format('delta')\
              .option('checkpointLocation', 'abfss://aldestination@adlsfordbs.dfs.core.windows.net/checkpoint')\
              .trigger(processingTime= '5 seconds')\
              .start('abfss://aldestination@adlsfordbs.dfs.core.windows.net/data')